# Fashion Item Classification with a CNN
**Domain:** Computer Vision / Retail  |  **Dataset:** Fashion-MNIST  |  **Framework:** TensorFlow / Keras

**Goal:** Train a small CNN that classifies 28x28 grayscale clothing images into 10 categories, so an online catalogue can tag product images automatically.

**Pipeline:** load data -> normalize -> train CNN -> evaluate (accuracy, macro F1, per-class metrics) -> confusion matrix -> inspect misclassifications -> explain confusions.

## 1. Setup and reproducibility

In [ ]:
import os, random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
try:
    tf.keras.utils.set_random_seed(SEED)          # seeds python, numpy, tf together
    tf.config.experimental.enable_op_determinism() # more reproducible (may be slower)
except Exception as e:
    print("Determinism option not available:", e)

print("TensorFlow:", tf.__version__)

## 2. Load Fashion-MNIST

In [ ]:
(x_train_raw, y_train), (x_test_raw, y_test) = keras.datasets.fashion_mnist.load_data()

class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

print("Train:", x_train_raw.shape, y_train.shape)
print("Test: ", x_test_raw.shape, y_test.shape)
print("Pixel range:", x_train_raw.min(), "-", x_train_raw.max(), "| dtype:", x_train_raw.dtype)
print("Train class counts:", np.bincount(y_train))

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for i, ax in enumerate(axes.ravel()):
    idx = np.where(y_train == i)[0][0]
    ax.imshow(x_train_raw[idx], cmap="gray")
    ax.set_title(class_names[i]); ax.axis("off")
plt.suptitle("One example per class"); plt.tight_layout(); plt.show()

## 3. Preprocessing
- Scale pixels from `[0, 255]` to `[0, 1]` (stable, faster gradient-based training).
- Add a channel dimension: `(N, 28, 28)` -> `(N, 28, 28, 1)`, which Conv2D expects.
- Hold out 10% of the training set as a **validation set** so the test set is only used once, at the end.

In [ ]:
x_train_full = (x_train_raw.astype("float32") / 255.0)[..., np.newaxis]
x_test       = (x_test_raw.astype("float32") / 255.0)[..., np.newaxis]

# Stratified validation split with a fixed seed
from sklearn.model_selection import train_test_split
x_train, x_val, y_train_s, y_val = train_test_split(
    x_train_full, y_train, test_size=0.1, stratify=y_train, random_state=SEED)

print("Train:", x_train.shape, "Val:", x_val.shape, "Test:", x_test.shape)
print("Value range after scaling:", x_train.min(), "-", x_train.max())

## 4. Build a small CNN
Two convolution blocks (Conv -> BatchNorm -> Conv -> MaxPool -> Dropout) followed by a compact dense head. Small enough to train in minutes on CPU, yet strong enough to reach roughly 93-94% test accuracy.

In [ ]:
def build_model():
    model = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),

        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Dropout(0.25),

        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Dropout(0.25),

        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(10, activation="softmax"),
    ])
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

model = build_model()
model.summary()

## 5. Train

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, verbose=1),
]

history = model.fit(
    x_train, y_train_s,
    validation_data=(x_val, y_val),
    epochs=25, batch_size=128,
    callbacks=callbacks, verbose=2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history["loss"], label="train"); ax[0].plot(history.history["val_loss"], label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("Epoch"); ax[0].legend()
ax[1].plot(history.history["accuracy"], label="train"); ax[1].plot(history.history["val_accuracy"], label="val")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("Epoch"); ax[1].legend()
plt.tight_layout(); plt.show()

## 6. Evaluate on the test set: accuracy and macro F1

In [ ]:
probs  = model.predict(x_test, verbose=0)
y_pred = probs.argmax(axis=1)

acc      = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")

print(f"Test accuracy : {acc:.4f}")
print(f"Macro F1-score: {macro_f1:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=class_names, digits=3))

In [ ]:
# ---- Consistency check: all numbers in the report must come from THIS run ----
n_wrong = int((y_pred != y_test).sum())
print(f"Misclassified images : {n_wrong} of {len(y_test)}  ({n_wrong/len(y_test):.2%})")
print(f"Expected from accuracy: {round(len(y_test)*(1-acc))}")
assert n_wrong == round(len(y_test) * (1 - acc)), "Mismatch - re-run the notebook from the top"
print("OK: accuracy, confusion matrix and misclassification counts agree.")

# Save this run's predictions so results can be re-checked later
np.save("y_pred_final.npy", y_pred)
np.save("y_test_final.npy", y_test)

In [ ]:
# Class-level comparison: F1 per class, sorted
f1_per_class = f1_score(y_test, y_pred, average=None)
order = np.argsort(f1_per_class)

plt.figure(figsize=(9, 4.5))
bars = plt.barh([class_names[i] for i in order], f1_per_class[order], color="steelblue")
plt.axvline(macro_f1, color="red", ls="--", label=f"macro F1 = {macro_f1:.3f}")
plt.xlim(0.7, 1.0); plt.xlabel("F1-score"); plt.title("Per-class F1 (worst to best)")
plt.legend(); plt.tight_layout(); plt.show()

print("Hardest classes:", [class_names[i] for i in order[:3]])
print("Easiest classes:", [class_names[i] for i in order[-3:]])

## 7. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False)
ax.set_title("Confusion matrix (counts)")
plt.tight_layout(); plt.show()

# Row-normalised version: share of each true class sent to each predicted class
cm_norm = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(cm_norm, display_labels=class_names).plot(
    ax=ax, cmap="Blues", xticks_rotation=45, colorbar=False, values_format=".2f")
ax.set_title("Confusion matrix (row-normalised)")
plt.tight_layout(); plt.show()

In [ ]:
# Top confused (true -> predicted) pairs
pairs = [(cm[i, j], class_names[i], class_names[j])
         for i in range(10) for j in range(10) if i != j]
pairs.sort(reverse=True)

print("Most frequent confusions:")
for n, t, p in pairs[:8]:
    print(f"  {t:12s} -> {p:12s}: {n} images")

## 8. Sample predictions

In [ ]:
def show_predictions(indices, title, cols=5):
    rows = int(np.ceil(len(indices) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.6, rows * 3.4))
    axes = np.atleast_1d(axes).ravel()
    for ax in axes:
        ax.axis("off")                      # also hides unused cells
    for ax, idx in zip(axes, indices):
        ax.imshow(x_test[idx].squeeze(), cmap="gray")
        pred, true = y_pred[idx], y_test[idx]
        color = "green" if pred == true else "red"
        ax.set_title(f"P: {class_names[pred]}\nT: {class_names[true]}\n({probs[idx, pred]:.0%})",
                     color=color, fontsize=9, pad=6)
    fig.suptitle(title, fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.95], h_pad=2.5)   # h_pad = vertical gap between rows
    plt.show()

rng = np.random.default_rng(SEED)
show_predictions(rng.choice(len(x_test), 15, replace=False),
                 "Random test predictions (green = correct, red = wrong)")

## 9. Inspect misclassified clothing

In [ ]:
wrong = np.where(y_pred != y_test)[0]
print(f"{len(wrong)} misclassified out of {len(y_test)} ({len(wrong)/len(y_test):.2%})")

# Most confident mistakes: the model was very sure and still "wrong".
# Many of these turn out to be odd images or questionable dataset labels.
conf_wrong = wrong[np.argsort(-probs[wrong].max(axis=1))]
show_predictions(conf_wrong[:15], "Most confident misclassifications")

In [ ]:
# Examples for the most confused pairs, side by side
def show_pair(true_c, pred_c, n=5):
    idx = np.where((y_test == true_c) & (y_pred == pred_c))[0][:n]
    if len(idx) == 0:
        return
    plt.figure(figsize=(n * 2.2, 2.6))
    for k, i in enumerate(idx):
        plt.subplot(1, n, k + 1)
        plt.imshow(x_test[i].squeeze(), cmap="gray"); plt.axis("off")
    plt.suptitle(f"True: {class_names[true_c]}  ->  Predicted: {class_names[pred_c]}")
    plt.tight_layout(); plt.show()

for _, t, p in pairs[:4]:
    show_pair(class_names.index(t), class_names.index(p))

## 9b. Business use: send low-confidence items to human review
In a real catalogue the model can tag confident predictions automatically and pass uncertain ones to a person. The table shows the trade-off between automation and accuracy.

In [ ]:
conf = probs.max(axis=1)
thresholds = [0.0, 0.5, 0.7, 0.8, 0.9, 0.95, 0.99]

print(f"{'threshold':>9} | {'auto-tagged':>11} | {'accuracy on auto-tagged':>23} | {'sent to review':>14}")
auto_share, auto_acc = [], []
for t in thresholds:
    keep = conf >= t
    a = (y_pred[keep] == y_test[keep]).mean()
    auto_share.append(keep.mean()); auto_acc.append(a)
    print(f"{t:>9.2f} | {keep.mean():>11.1%} | {a:>23.2%} | {1-keep.mean():>14.1%}")

plt.figure(figsize=(7, 4))
plt.plot(auto_share, auto_acc, marker="o")
for t, x, y in zip(thresholds, auto_share, auto_acc):
    plt.annotate(f"{t:.2f}", (x, y), textcoords="offset points", xytext=(5, 5), fontsize=8)
plt.xlabel("Share of images tagged automatically"); plt.ylabel("Accuracy on those images")
plt.title("Automation vs accuracy (labels = confidence threshold)")
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 10. Why are some classes confused?

Fashion-MNIST images are only 28x28 pixels, grayscale and centred, so colour, texture, fabric, buttons, collars and logos are largely lost. The model can only rely on **silhouette and coarse shape**. Classes with overlapping silhouettes are therefore hardest.

*(Counts below are from the final run: 93.88% accuracy, 612 errors. Re-training may shift them by a few images.)*

### Main finding: Shirt is the problem class
- Shirt has the lowest F1 (0.805; recall 0.782, precision 0.828). Every other class is at 0.89 or higher.
- **380 of the 612 errors (62%) involve Shirt**: 218 real shirts are misclassified, and 162 other items are wrongly predicted as Shirt.
- **533 of the 612 errors (87%)** start in just five classes: T-shirt/top, Pullover, Dress, Coat and Shirt.

| Confusion (true -> predicted) | Count | Why it happens |
|---|---|---|
| **Shirt -> T-shirt/top** | 98 | Same torso outline; collars and buttons are only a few pixels wide. Many images labelled "Shirt" are short-sleeved. |
| **T-shirt/top -> Shirt** | 67 | The reverse: some "T-shirts" are long-sleeved or have a button placket. The two directions are not equal, because "Shirt" is the more varied class. |
| **Shirt -> Coat** | 51 | Plain long-sleeved shirts or shirt-jackets look like light coats without colour or a visible collar. |
| **Shirt -> Pullover** | 44 | Long-sleeved tops with near-identical shape. |
| **Coat -> Shirt** | 40 | Same overlap in the other direction. |
| **Pullover -> Shirt** | 30 | Same overlap in the other direction. |
| **Coat <-> Pullover** | 30 and 25 | Similar sleeves and body; coats differ mainly in fine details such as openings and pockets. |
| **Shirt <-> Dress** | 22 and 22 | Long, narrow tops can resemble short dresses. |
| **Ankle boot <-> Sneaker** | 20 and 18 | Low-cut boots look like sneakers from the side. |

### What stays easy
Trouser, Bag, Sandal, Sneaker and Ankle boot all have F1 of about 0.97 or higher because their silhouettes are distinctive. Footwear is almost never confused with clothing: only 58 errors in total across the three footwear classes, nearly all of them between footwear classes.

### Two kinds of error
1. **Boundary ambiguity** (Shirt / T-shirt / Pullover / Coat): moderate confidence, caused by overlapping class definitions and low resolution.
2. **Label noise or unusual images** (e.g. a high-heeled shoe labelled "Ankle boot" and predicted "Sandal"; a dress-like garment labelled "Trouser"): the model is about 100% confident, and its answer often looks more plausible than the dataset label. Part of the remaining error is therefore not the model's fault, and a perfect score is not achievable on this dataset.

### Other contributing factors
- *Intra-class variation:* "Shirt" covers folded, long, short, patterned and cropped items.
- *Low resolution:* fine details needed to separate these classes are not present in the data.

## 11. Ideas to improve
- Data augmentation (small shifts, rotations, horizontal flips) to improve generalisation.
- Deeper or residual architecture, class weights for Shirt, or an ensemble of several seeds.
- Higher-resolution colour images, where collars, buttons and textures are visible.
- Cleaning or re-checking suspicious labels in the training data.
- A confidence threshold with human review for upper-body clothing (see Section 9b).

In [ ]:
shirt = class_names.index("Shirt")
errors = (y_pred != y_test)
shirt_involved = errors & ((y_test == shirt) | (y_pred == shirt))
share = shirt_involved.sum() / errors.sum()

print("=" * 60)
print("FINAL RESULTS")
print("=" * 60)
print(f"Test accuracy        : {acc:.4f}")
print(f"Macro F1-score       : {macro_f1:.4f}")
print(f"Total errors         : {errors.sum()} of {len(y_test)}")
print(f"Weakest class        : {class_names[order[0]]} (F1 = {f1_per_class[order[0]]:.3f})")
print(f"Strongest class      : {class_names[order[-1]]} (F1 = {f1_per_class[order[-1]]:.3f})")
print(f"Errors involving Shirt: {share:.0%}")
print()
print("CONCLUSION")
print(f"The CNN achieved {acc:.1%} test accuracy and a macro F1 of {macro_f1:.3f}. "
      f"{class_names[order[0]]} was the weakest class (F1 = {f1_per_class[order[0]]:.2f}); "
      f"about {share:.0%} of all errors involved Shirt, mainly confused with T-shirt/top, "
      f"Coat and Pullover. These garments share near-identical silhouettes at 28x28 grayscale, "
      f"and some confident errors appear to be mislabelled or atypical images. "
      f"For a real catalogue, low-confidence upper-body predictions should go to human review.")

## 12. Save the model for the demo app
Saves the trained model (about 2 MB) and a small set of sample images, then downloads both. Put `fashion_cnn.keras` and the unzipped `samples/` folder next to `app.py` on your computer.

In [ ]:
import os, zipfile
from PIL import Image

model.save("fashion_cnn.keras")
print("Saved fashion_cnn.keras  (%.1f MB)" % (os.path.getsize("fashion_cnn.keras") / 1e6))

# Sample images for the demo: 3 correctly predicted per class + 6 real mistakes
os.makedirs("samples", exist_ok=True)
def save_img(i, name):
    Image.fromarray(x_test_raw[i]).resize((280, 280), Image.NEAREST).save(f"samples/{name}.png")

for c in range(10):
    ok = np.where((y_test == c) & (y_pred == c))[0][:3]
    for k, i in enumerate(ok):
        save_img(i, f"{class_names[c].replace('/', '-').replace(' ', '_')}_{k+1}")

shirt_to_tshirt = np.where((y_test == 6) & (y_pred == 0))[0][:6]
for k, i in enumerate(shirt_to_tshirt):
    save_img(i, f"WRONG_Shirt_predicted_Tshirt_{k+1}")

with zipfile.ZipFile("samples.zip", "w") as z:
    for f in sorted(os.listdir("samples")):
        z.write(os.path.join("samples", f))
print("Saved", len(os.listdir("samples")), "sample images -> samples.zip")

try:
    from google.colab import files
    files.download("fashion_cnn.keras")
    files.download("samples.zip")
except ImportError:
    print("Not running in Colab: files are in the current folder.")